# Lesson 01 - Introduction to AI Agents

Welcome to the first lesson in the **AI Agents for Beginners** course!

An **AI agent** is a program that uses a large language model (LLM) as its reasoning engine and can take *actions* in the real world — calling APIs, querying databases, or running code — to accomplish a goal on behalf of a user.

In this notebook you will build your first agent: a **Travel Agent** that recommends vacation destinations. Along the way you will learn how to:

1. Connect to Microsoft Foundry Agent Service using the **Microsoft Agent Framework**.
2. Give the agent a **tool** — a plain Python function it can call.
3. Run the agent and inspect its response.
4. Stream the agent's response token-by-token.

## Setup

Before running this notebook, make sure you have:

1. **A Microsoft Foundry project** with a deployed chat model (e.g. `gpt-5-mini`).
2. **Logged in with the Azure CLI** — run `az login` in your terminal.
3. **Set the required environment variables:**
   - `AZURE_AI_PROJECT_ENDPOINT` — your Microsoft Foundry project endpoint.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — the name of your deployed model.

The cell below installs the Python packages you need.

In [1]:
%pip install agent-framework azure-ai-projects azure-identity -q

Note: you may need to restart the kernel to use updated packages.


In [5]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Creating Your First Agent

An agent needs two things:

- **Instructions** that tell it *who it is* and *how to behave* (a system prompt).
- **Tools** — Python functions decorated with `@tool` that the agent can call to retrieve information or perform actions.

Below we define a simple tool that returns a list of popular vacation destinations. The agent will use this tool when a user asks for travel recommendations.

In [6]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [8]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

Great — from the destinations I can see, here are the warm beach options I recommend, with a quick note on why each might fit depending on the vibe you want:

1. Bali (Indonesia)
- Vibe: Tropical, relaxed, surf and yoga, rice terraces and temples nearby.
- Highlights: Seminyak/Canggu beaches, Uluwatu surf, snorkeling/diving around Nusa Lembongan/penida.
- Best time: Apr–Oct (drier).
- Good for: Culture + beach combos, affordable luxury, surfing/snorkeling.

2. Rio de Janeiro (Brazil)
- Vibe: Lively, iconic beaches, nightlife and samba.
- Highlights: Copacabana, Ipanema, Sugarloaf, nightlife in Lapa.
- Best time: Year-round beach weather; Dec–Mar is hottest.
- Good for: Party atmosphere, city+beach experience, dramatic scenery.

3. Barcelona (Spain)
- Vibe: Urban-beach mix — city life, tapas, architecture plus Mediterranean beaches.
- Highlights: Barceloneta, quieter beaches a short train ride away, great food and culture.
- Best time: May–Sep for warm beach weather.
- Good for: Easy sh

## Streaming Responses

For a more interactive experience you can **stream** the agent's response. Instead of waiting for the full reply, the agent yields text chunks as they are generated. This is especially useful in chat interfaces where you want to display output in real time.

In [9]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

Great choice — Tokyo is a fascinating, hyperactive city that mixes cutting‑edge modernity with pockets of deep tradition. Below is a compact guide to help you decide if it fits your ideal trip and how to plan one.

Quick snapshot
- Vibe: Ultra-modern skyscrapers, neon nightlife, world‑class food, tranquil shrines, and highly efficient public transit.
- Best for: Food lovers, culture seekers, shoppers, tech/Anime fans, photographers, and first‑time visitors to Japan.
- Best time to go: Spring (late March–April) for cherry blossoms and autumn (October–November) for foliage. Summer is hot and humid; winter is cool and less crowded.

Top sights & neighborhoods
- Shinjuku: Major transport hub, nightlife (Kabukicho), Metropolitan Government Building observatory.
- Shibuya & Harajuku: Famous crossing, youth fashion, Takeshita Street, Meiji Shrine.
- Ginza: Upscale shopping, department stores, fine dining.
- Asakusa: Senso‑ji temple, traditional shops, good for budget stays.
- Akihabara: Elect

## Summary

In this lesson you learned how to:

- **Create a provider** that connects to Microsoft Foundry Agent Service via `FoundryChatClient`.
- **Define a tool** using the `@tool` decorator so the agent can call your Python functions.
- **Run the agent** with a user message and print its response.
- **Stream responses** for real-time output.

In the next lesson we will explore agentic frameworks in more depth and learn how to give agents more powerful tools and multi-step reasoning capabilities.